# Notebook 11 — Ranking Feature Engineering & Ablation

**Goal:** Engineer history-only features and measure their incremental value for LightGBM ranking on a fixed Notebook 08 Top-100 candidate pool. Model selection uses the August **user-disjoint holdout only**; September test is evaluated once after selecting the feature group. This is implicit-feedback recommendation, **not observed ad CTR**.

## 0. Setup
Run from project root or `notebooks/`. Dependencies: `pip install pandas numpy scikit-learn lightgbm`. This notebook reconstructs the Notebook 09 baseline features to keep definitions comparable.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

cwd=Path.cwd().resolve()
if (cwd/'data'/'processed').is_dir(): ROOT=cwd
elif (cwd.parent/'data'/'processed').is_dir(): ROOT=cwd.parent
else: raise FileNotFoundError('Cannot locate project data/processed')
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
import lightgbm as lgb
from sklearn.model_selection import train_test_split

SEED=42; K_VALUES=[10,20,50,100]; MAX_K=100
CUT={'val':pd.Timestamp('2015-08-01',tz='UTC'), 'test':pd.Timestamp('2015-09-01',tz='UTC')}
raw={}
for split,stem,hfile in [('val','validation','train_history.csv'),('test','test','test_history.csv')]:
    cand=pd.read_csv(PROC/f'notebook08_{split}_hybrid_candidates.csv',dtype={'visitorid':'int64','rank':'int16','itemid':'int64'})
    hist=pd.read_csv(PROC/hfile,usecols=['timestamp','visitorid','event','itemid'])
    labels=pd.read_csv(PROC/f'{stem}_view_labels_warm_users.csv',usecols=['visitorid','itemid'])
    assert hist.timestamp.max()<int(CUT[split].timestamp()*1000)
    assert not cand[['visitorid','itemid']].duplicated().any()
    assert not cand[['visitorid','rank']].duplicated().any()
    assert cand.groupby('visitorid').size().eq(MAX_K).all()
    assert set(cand.visitorid.unique())==set(labels.visitorid.unique())
    assert set(['from_itemcf','from_faiss_fallback','from_popularity']).issubset(cand.columns)
    raw[split]=(cand,hist,labels)
    print(split,'candidate rows',len(cand),'history',len(hist),'positive labels',len(labels))

val candidate rows 1332900 history 1899029 positive labels 28634
test candidate rows 1513700 history 2452391 positive labels 31876


## 1. Rebuild the 15 baseline features
All item/user aggregates use the corresponding historical cutoff, never future labels.

In [2]:
FEATURES=['retrieval_rank','rrf_inverse_rank','from_itemcf','from_faiss_fallback','from_popularity',
          'source_count','log_item_views','log_item_unique_viewers','log_user_views',
          'log_user_unique_items','user_last_event_days','item_last_view_days',
          'item_view_share','log_item_carts','log_item_purchases']

def make_features(candidates, history, cutoff):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    stats=views.groupby('itemid',sort=False).agg(item_views=('visitorid','size'),
          item_unique_viewers=('visitorid','nunique'),item_last_view=('timestamp','max'))
    users=views.groupby('visitorid',sort=False).agg(user_views=('itemid','size'),
          user_unique_items=('itemid','nunique'),user_last_event=('timestamp','max'))
    other=history.loc[history.event.isin(['addtocart','transaction']),['itemid','event']]
    if len(other):
        event_counts=other.groupby(['itemid','event']).size().unstack(fill_value=0)
        stats['item_carts']=event_counts['addtocart'] if 'addtocart' in event_counts else 0
        stats['item_purchases']=event_counts['transaction'] if 'transaction' in event_counts else 0
    else: stats['item_carts']=0;stats['item_purchases']=0
    stats[['item_carts','item_purchases']]=stats[['item_carts','item_purchases']].fillna(0)
    d=candidates.merge(stats,left_on='itemid',right_index=True,how='left',validate='many_to_one')
    d=d.merge(users,left_on='visitorid',right_index=True,how='left',validate='many_to_one')
    ms=int(cutoff.timestamp()*1000)
    d['retrieval_rank']=d['rank'].astype('float32')
    d['rrf_inverse_rank']=(1.0/(60+d['rank'])).astype('float32')
    d['source_count']=d[['from_itemcf','from_faiss_fallback','from_popularity']].sum(axis=1)
    for src,dst in [('item_views','log_item_views'),('item_unique_viewers','log_item_unique_viewers'),
                    ('user_views','log_user_views'),('user_unique_items','log_user_unique_items'),
                    ('item_carts','log_item_carts'),('item_purchases','log_item_purchases')]:
        d[dst]=np.log1p(d[src].fillna(0)).astype('float32')
    d['user_last_event_days']=((ms-d.user_last_event)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_last_view_days']=((ms-d.item_last_view)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_view_share']=(d.item_views.fillna(0)/max(1,len(views))).astype('float32')
    for name in FEATURES: d[name]=d[name].fillna(0).astype('float32')
    return d[['visitorid','itemid','rank']+FEATURES].copy()


## 2. Additional history-only features
New signals: short-window popularity, trend, user event mix, user recency, item event rates, item-vs-user activity interactions. Do **not** use any future outcome labels to compute features.

In [3]:
EXTRA_FEATURES=[
    'log_item_views_7d','log_item_views_30d','item_recent_7d_share','item_trend_7d_vs_30d',
    'log_item_unique_viewers_30d','item_cart_per_view','item_purchase_per_view',
    'log_user_carts','log_user_purchases','user_cart_per_view','user_purchase_per_view',
    'user_views_7d','user_views_30d','user_recent_7d_share',
    'user_item_activity_interaction','item_popularity_user_activity_ratio',
    'item_recency_user_recency_gap','source_itemcf_x_recency','source_faiss_x_trend'
]
TEMPORAL_FEATURES=['log_item_views_7d','log_item_views_30d','item_recent_7d_share',
                   'item_trend_7d_vs_30d','log_item_unique_viewers_30d',
                   'user_views_7d','user_views_30d','user_recent_7d_share',
                   'item_recency_user_recency_gap','source_itemcf_x_recency','source_faiss_x_trend']
SOURCE_FEATURES=['from_itemcf','from_faiss_fallback','from_popularity','source_count',
                 'source_itemcf_x_recency','source_faiss_x_trend']

def add_extra_features(baseline,history,cutoff):
    d=baseline.copy()
    cutoff_ms=int(cutoff.timestamp()*1000)
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    recent7=views.loc[views.timestamp>=cutoff_ms-7*86400000]
    recent30=views.loc[views.timestamp>=cutoff_ms-30*86400000]
    item7=recent7.groupby('itemid').size().rename('iv7')
    item30=recent30.groupby('itemid').agg(iv30=('visitorid','size'),uv30=('visitorid','nunique'))
    user7=recent7.groupby('visitorid').size().rename('uv7')
    user30=recent30.groupby('visitorid').size().rename('uv30')
    user_events=(history.loc[history.event.isin(['addtocart','transaction'])]
                 .groupby(['visitorid','event']).size().unstack(fill_value=0))
    item_events=(history.loc[history.event.isin(['addtocart','transaction'])]
                 .groupby(['itemid','event']).size().unstack(fill_value=0))
    item_total=views.groupby('itemid').size().rename('ivall')
    user_total=views.groupby('visitorid').size().rename('uvall')
    def lookup(ids,series):
        return ids.map(series).fillna(0).to_numpy(dtype=np.float32)
    item_ids=d.itemid; user_ids=d.visitorid
    iv7=lookup(item_ids,item7)
    iv30=lookup(item_ids,item30['iv30'])
    uv30item=lookup(item_ids,item30['uv30'])
    ivall=lookup(item_ids,item_total)
    uv7=lookup(user_ids,user7)
    uv30=lookup(user_ids,user30)
    uvall=lookup(user_ids,user_total)
    def event_lookup(table,name,ids):
        return lookup(ids,table[name]) if name in table.columns else np.zeros(len(ids),dtype=np.float32)
    ic=event_lookup(item_events,'addtocart',item_ids)
    ip=event_lookup(item_events,'transaction',item_ids)
    uc=event_lookup(user_events,'addtocart',user_ids)
    up=event_lookup(user_events,'transaction',user_ids)
    d['log_item_views_7d']=np.log1p(iv7)
    d['log_item_views_30d']=np.log1p(iv30)
    d['item_recent_7d_share']=iv7/(ivall+1)
    d['item_trend_7d_vs_30d']=(iv7+1)/(iv30/30*7+1)
    d['log_item_unique_viewers_30d']=np.log1p(uv30item)
    d['item_cart_per_view']=ic/(ivall+1)
    d['item_purchase_per_view']=ip/(ivall+1)
    d['log_user_carts']=np.log1p(uc)
    d['log_user_purchases']=np.log1p(up)
    d['user_cart_per_view']=uc/(uvall+1)
    d['user_purchase_per_view']=up/(uvall+1)
    d['user_views_7d']=np.log1p(uv7)
    d['user_views_30d']=np.log1p(uv30)
    d['user_recent_7d_share']=uv7/(uvall+1)
    d['user_item_activity_interaction']=d.log_user_views.to_numpy()*d.log_item_views.to_numpy()
    d['item_popularity_user_activity_ratio']=d.log_item_views.to_numpy()/(1+d.log_user_views.to_numpy())
    d['item_recency_user_recency_gap']=np.clip(d.item_last_view_days.to_numpy()-d.user_last_event_days.to_numpy(),-365,365)
    d['source_itemcf_x_recency']=d.from_itemcf.to_numpy()/(1+d.item_last_view_days.to_numpy())
    d['source_faiss_x_trend']=d.from_faiss_fallback.to_numpy()*d.item_trend_7d_vs_30d.to_numpy()
    d[EXTRA_FEATURES]=d[EXTRA_FEATURES].replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
    assert d[EXTRA_FEATURES].notna().all().all()
    return d

frames={}
for split in ('val','test'):
    cand,hist,labels=raw[split]
    t=time.perf_counter()
    baseline=make_features(cand,hist,CUT[split])
    frames[split]=add_extra_features(baseline,hist,CUT[split])
    print(split,'shape',frames[split].shape,'seconds',round(time.perf_counter()-t,2))
assert len(EXTRA_FEATURES)==len(set(EXTRA_FEATURES))
assert not set(EXTRA_FEATURES)&set(FEATURES)


val shape (1332900, 37) seconds 4.13
test shape (1513700, 37) seconds 4.91


## 3. Labels and deterministic user-disjoint holdout
The labels are attached **only after** feature creation. Negatives are unobserved candidate pairs, not confirmed non-click impressions.

In [4]:
def attach_labels(df, labels):
    positives=labels[['visitorid','itemid']].drop_duplicates().assign(label=np.int8(1))
    out=df.merge(positives,on=['visitorid','itemid'],how='left',validate='one_to_one')
    out['label']=out['label'].fillna(0).astype('int8')
    return out

val=attach_labels(frames['val'],raw['val'][2]);test=attach_labels(frames['test'],raw['test'][2])
all_users=np.sort(val.visitorid.unique())
train_users,holdout_users=train_test_split(all_users,test_size=.20,random_state=SEED)
train_users=set(map(int,train_users));holdout_users=set(map(int,holdout_users))
assert not(train_users & holdout_users)

# Deterministic negative sampling by user; retain all positives and at most 20 implicit negatives.
def sample_train_rows(frame, user_ids, neg_per_user=20, seed=SEED):
    sub=frame.loc[frame.visitorid.isin(user_ids)].copy()
    positives=sub[sub.label.eq(1)]
    negatives=sub[sub.label.eq(0)]
    negatives=(negatives.assign(_random=np.random.default_rng(seed).random(len(negatives)))
              .sort_values(['visitorid','_random'],kind='mergesort')
              .groupby('visitorid',sort=False).head(neg_per_user).drop(columns='_random'))
    return pd.concat([positives,negatives],ignore_index=True).sort_values(['visitorid','rank'],kind='mergesort').reset_index(drop=True)

train_df=sample_train_rows(val,train_users)
valid_df=val.loc[val.visitorid.isin(holdout_users)].sort_values(['visitorid','rank']).reset_index(drop=True)
assert train_df.visitorid.nunique()==len(train_users)
print('Training candidates:',len(train_df),'positive:',int(train_df.label.sum()))
print('Held-out candidates:',len(valid_df),'positive:',int(valid_df.label.sum()))
print('Test candidates:',len(test),'positive:',int(test.label.sum()))

Training candidates: 214772 positive: 1512
Held-out candidates: 266600 positive: 376
Test candidates: 1513700 positive: 1829


## 4. Holdout ablations (no September labels used for selection)
Compare baseline 15 features, enhanced features, no source flags, and no temporal features. Select on August holdout Recall@10, tie-break on Recall@20.

In [5]:
def rank_candidates(frame, scores):
    d=frame[['visitorid','itemid','rank']].copy()
    d['score']=np.asarray(scores,dtype='float32')
    d=d.sort_values(['visitorid','score','rank','itemid'],ascending=[True,False,True,True],kind='mergesort')
    d['new_rank']=d.groupby('visitorid',sort=False).cumcount()+1
    return d

def evaluate_ranks(frame, labels, rank_column, k_values=K_VALUES):
    # Use complete label sets, not only positives present in candidate table.
    truth=labels.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    f=frame.sort_values(['visitorid',rank_column],kind='mergesort')
    recs=f.groupby('visitorid',sort=False).itemid.agg(list).to_dict()
    assert set(recs)==set(truth)
    eligible_hits=sum(len(set(items)&truth[int(u)]) for u,items in recs.items())
    rows=[]
    for k in k_values:
        found=0;hit_users=0;macro=[]
        for u,items in recs.items():
            hits=len(set(items[:k])&truth[int(u)])
            found+=hits;hit_users+=int(hits>0);macro.append(hits/len(truth[int(u)]))
        rows.append(dict(K=k,Recall_micro=found/sum(map(len,truth.values())),
                         Recall_macro=float(np.mean(macro)),HitRate=hit_users/len(truth),
                         CandidateConditionedRecall=found/eligible_hits if eligible_hits else np.nan,
                         hits=found))
    return pd.DataFrame(rows)


GROUPS={
    'baseline15':FEATURES,
    'enhanced34':FEATURES+EXTRA_FEATURES,
    'without_source': [f for f in FEATURES+EXTRA_FEATURES if f not in SOURCE_FEATURES],
    'without_temporal': [f for f in FEATURES+EXTRA_FEATURES if f not in TEMPORAL_FEATURES],
}
params=dict(objective='binary',learning_rate=.05,num_leaves=31,max_depth=-1,
            min_child_samples=50,colsample_bytree=.9,subsample=.9,subsample_freq=1,
            reg_lambda=2.,random_state=SEED,n_jobs=-1,verbosity=-1)
val_labels=raw['val'][2].loc[raw['val'][2].visitorid.isin(holdout_users)]
base_metrics=evaluate_ranks(valid_df,val_labels,'rank')
models={};results={};importance_tables=[]
for name,cols in GROUPS.items():
    model=lgb.LGBMClassifier(n_estimators=600,**params)
    t=time.perf_counter()
    model.fit(train_df[cols],train_df.label,eval_set=[(valid_df[cols],valid_df.label)],
              eval_metric='binary_logloss',callbacks=[lgb.early_stopping(35,verbose=False)])
    pred=model.predict_proba(valid_df[cols])[:,1]
    ranked=rank_candidates(valid_df,pred)
    metrics=evaluate_ranks(ranked,val_labels,'new_rank')
    models[name]=model;results[name]=metrics
    importance_tables.append(pd.DataFrame({'model':name,'feature':cols,'importance':model.feature_importances_}))
    print(name,'features',len(cols),'best_iteration',model.best_iteration_,
          'holdout Recall@10',round(float(metrics.iloc[0].Recall_micro),6),
          'seconds',round(time.perf_counter()-t,1))
comparison=pd.DataFrame({'K':K_VALUES,'hybrid':base_metrics.Recall_micro.to_numpy(),
                          **{k:v.Recall_micro.to_numpy() for k,v in results.items()}})
display(comparison)
chosen=max(GROUPS,key=lambda x:(float(results[x].iloc[0].Recall_micro),float(results[x].iloc[1].Recall_micro)))
print('Selected feature group on August holdout:',chosen)
importance=pd.concat(importance_tables,ignore_index=True)


baseline15 features 15 best_iteration 427 holdout Recall@10 0.03227 seconds 4.2
enhanced34 features 34 best_iteration 594 holdout Recall@10 0.032463 seconds 6.5
without_source features 28 best_iteration 600 holdout Recall@10 0.029005 seconds 5.7
without_temporal features 23 best_iteration 600 holdout Recall@10 0.031886 seconds 5.5


,K,hybrid,baseline15,enhanced34,without_source,without_temporal
0,10,0.025740,0.032270,0.032463,0.029005,0.031886
1,20,0.038033,0.046101,0.044756,0.041106,0.043988
2,50,0.054552,0.061468,0.062428,0.059931,0.061852
3,100,0.072224,0.072224,0.072224,0.072224,0.072224


Selected feature group on August holdout: enhanced34


## 5. Refit selected group on all August users and evaluate September **once**
Do not change selected features or hyperparameters after inspecting September results.

In [6]:
all_aug=sample_train_rows(val,set(map(int,all_users)),neg_per_user=20)
selected_features=GROUPS[chosen]
selected_iteration=max(1,int(models[chosen].best_iteration_))
final_model=lgb.LGBMClassifier(n_estimators=selected_iteration,**params)
t0=time.perf_counter()
final_model.fit(all_aug[selected_features],all_aug.label)
print('Full August refit seconds:',round(time.perf_counter()-t0,2))
t0=time.perf_counter()
test_scores=final_model.predict_proba(test[selected_features])[:,1]
test_scoring_seconds=time.perf_counter()-t0
test_ranked=rank_candidates(test,test_scores)
base_test=evaluate_ranks(test,raw['test'][2],'rank')
new_test=evaluate_ranks(test_ranked,raw['test'][2],'new_rank')
test_comparison=base_test.merge(new_test,on='K',suffixes=('_hybrid','_feature_model'))
assert np.isclose(test_comparison.iloc[-1].Recall_micro_hybrid,test_comparison.iloc[-1].Recall_micro_feature_model)
print('Test scoring seconds:',round(test_scoring_seconds,3))
display(test_comparison[['K','Recall_micro_hybrid','Recall_micro_feature_model','HitRate_hybrid','HitRate_feature_model']])

lgb_path=REPORTS/'notebook09_test_metrics.csv'
deep_path=REPORTS/'notebook10_test_metrics.csv'
compare=test_comparison[['K','Recall_micro_hybrid','Recall_micro_feature_model']].copy()
if lgb_path.exists():
    ref=pd.read_csv(lgb_path)
    compare=compare.merge(ref[['K','Recall_micro_lightgbm']],on='K',how='left')
if deep_path.exists():
    ref=pd.read_csv(deep_path)
    compare=compare.merge(ref[['K','Recall_micro_deepfm']],on='K',how='left')
print('Reference models (read-only; NOT used for selection):')
display(compare)


Full August refit seconds: 5.14
Test scoring seconds: 7.967


,K,Recall_micro_hybrid,Recall_micro_feature_model,HitRate_hybrid,HitRate_feature_model
0,10,0.020768,0.026572,0.037590,0.047698
1,20,0.029427,0.036517,0.051133,0.061373
2,50,0.045018,0.049881,0.073198,0.079408
3,100,0.057379,0.057379,0.088855,0.088855


Reference models (read-only; NOT used for selection):


,K,Recall_micro_hybrid,Recall_micro_feature_model,Recall_micro_lightgbm,Recall_micro_deepfm
0,10,0.020768,0.026572,0.026383,0.027105
1,20,0.029427,0.036517,0.036767,0.037740
2,50,0.045018,0.049881,0.049881,0.050540
3,100,0.057379,0.057379,0.057379,0.057379


## 6. Save report, features, model and candidates
All outputs are saved at project root under `reports/` or `data/processed/`.

In [7]:
import pickle
with open(REPORTS/'notebook11_feature_engineered_lightgbm.pkl','wb') as f:
    pickle.dump({'model':final_model,'features':selected_features,'selected_group':chosen,
                 'training_cutoff':'2015-08-01','implicit_feedback':True},f)
test_ranked.to_csv(PROC/'notebook11_test_scored_candidates.csv',index=False)
comparison.to_csv(REPORTS/'notebook11_val_holdout_ablation.csv',index=False)
test_comparison.to_csv(REPORTS/'notebook11_test_metrics.csv',index=False)
compare.to_csv(REPORTS/'notebook11_test_model_comparison.csv',index=False)
importance.to_csv(REPORTS/'notebook11_feature_importance.csv',index=False)

def records(df):
    return json.loads(df.to_json(orient='records'))
report={'method':'history-only ranking feature engineering with LightGBM ablations',
        'protocol':'August user-disjoint holdout selects feature group; full-Aug refit; September test once',
        'caveats':['No impression data; sampled implicit negatives are not confirmed negatives',
                   'Not calibrated CTR/CVR','Fixed Notebook 08 Top-100 candidates',
                   'Selection uses August holdout, not September test'],
        'baseline_features':FEATURES,'new_features':EXTRA_FEATURES,
        'feature_groups':GROUPS,'selected_group':chosen,
        'selected_best_iteration':selected_iteration,
        'train_rows':len(train_df),'train_positives':int(train_df.label.sum()),
        'validation_holdout_users':len(holdout_users),
        'validation_holdout_ablation':records(comparison),
        'validation_holdout_metrics':{k:records(v) for k,v in results.items()},
        'test':records(test_comparison),'test_model_comparison':records(compare),
        'test_scoring_seconds':round(test_scoring_seconds,4)}
path=REPORTS/'notebook11_feature_engineering.json'
path.write_text(json.dumps(report,indent=2),encoding='utf-8')
assert path.exists() and (PROC/'notebook11_test_scored_candidates.csv').exists()
assert len(test_ranked)==len(test)
assert test_ranked.groupby('visitorid').size().eq(MAX_K).all()
print('PASS: historical feature boundaries, holdout selection, fixed candidate pool, report and model exports')
print('JSON:',path)


PASS: historical feature boundaries, holdout selection, fixed candidate pool, report and model exports
JSON: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook11_feature_engineering.json


## Interpretation and next stage
- Compare August holdout ablations first; use September as final check.
- Feature importance is predictive association, not causal evidence.
- Model scores are not calibrated CTR.
- Next: **Notebook 12 — Diversity-aware MMR Re-ranking**, with ranking relevance vs diversity tradeoff.